In [0]:
config = spark.table("anac.params.global_config").collect()[0]

gold_path = config["gold_path"]

date_format_br = config["date_format_br"]
date_hour_format_br = config["date_hour_format_br"]

In [0]:
df_gold = spark.sql("""
                    
select 
    Aerodromo_de_Destino, 
    Aerodromo_de_Origem,
    `Classificacao_da_Ocorrência`, 
    Danos_a_Aeronave, 
    Data_da_Ocorrencia,
    `Hora_da_Ocorrência`,
    Municipio,
    UF,
    Regiao,
    Tipo_de_Aerodromo, 
    Tipo_de_Ocorrencia,
    Lesoes_Desconhecidas_Passageiros, 
    Lesoes_Desconhecidas_Terceiros, 
    Lesoes_Desconhecidas_Tripulantes, 
    Lesoes_Fatais_Passageiros, 
    Lesoes_Fatais_Terceiros, 
    Lesoes_Fatais_Tripulantes, 
    Lesoes_Graves_Passageiros, 
    Lesoes_Graves_Terceiros, 
    Lesoes_Graves_Tripulantes, 
    Lesoes_Leves_Passageiros, 
    Lesoes_Leves_Terceiros, 
    Lesoes_Leves_Tripulantes
from anac.analytics.silver_ocorrencia_ampla

""")

In [0]:
colunas_somar = [
    "Lesoes_Desconhecidas_Passageiros",
    "Lesoes_Desconhecidas_Terceiros",
    "Lesoes_Desconhecidas_Tripulantes",
    "Lesoes_Fatais_Passageiros",
    "Lesoes_Fatais_Terceiros",
    "Lesoes_Fatais_Tripulantes",
    "Lesoes_Graves_Passageiros",
    "Lesoes_Graves_Terceiros",
    "Lesoes_Graves_Tripulantes",
    "Lesoes_Leves_Passageiros",
    "Lesoes_Leves_Terceiros",
    "Lesoes_Leves_Tripulantes"
]

df_gold = df_gold.withColumn("Lesões Total", sum(df_gold[somartudo] for somartudo in colunas_somar))

In [0]:
df_gold = df_gold \
    .withColumnRenamed('Aerodromo_de_Destino', 'Destino') \
    .withColumnRenamed('Aerodromo_de_Origem', 'Origem') \
    .withColumnRenamed('Classificacao_da_Ocorrência', 'Classificação') \
    .withColumnRenamed('Danos_a_Aeronave', 'Danos') \
    .withColumnRenamed('Data_da_Ocorrencia', 'Data') \
    .withColumnRenamed('Hora_da_Ocorrência', 'Hora') \
    .withColumnRenamed('Municipio', 'Município') \
    .withColumnRenamed('UF', 'Estado') \
    .withColumnRenamed('Regiao', 'Região') \
    .withColumnRenamed('Tipo_de_Aerodromo', 'Tipo Aerodromo') \
    .withColumnRenamed('Tipo_de_Ocorrencia', 'Tipo Ocorrência') \
    .withColumnRenamed('Lesoes_Desconhecidas_Passageiros', 'Lesões Desconhecidas Passageiros') \
    .withColumnRenamed('Lesoes_Desconhecidas_Terceiros', 'Lesões Desconhecidas Terceiros') \
    .withColumnRenamed('Lesoes_Desconhecidas_Tripulantes', 'Lesões Desconhecidas Tripulantes') \
    .withColumnRenamed('Lesoes_Fatais_Passageiros', 'Lesões Fatais Passageiros') \
    .withColumnRenamed('Lesoes_Fatais_Terceiros', 'Lesões Fatais Terceiros') \
    .withColumnRenamed('Lesoes_Fatais_Tripulantes', 'Lesões Fatais Tripulantes') \
    .withColumnRenamed('Lesoes_Graves_Passageiros', 'Lesões Graves_Passageiros') \
    .withColumnRenamed('Lesoes_Graves_Terceiros', 'Lesões Graves Terceiros') \
    .withColumnRenamed('Lesoes_Graves_Tripulantes', 'Lesões Graves Tripulantes') \
    .withColumnRenamed('Lesoes_Leves_Passageiros', 'Lesões Leves Passageiros') \
    .withColumnRenamed('Lesoes_Leves_Terceiros', 'Lesões Leves Terceiros') \
    .withColumnRenamed('Lesoes_Leves_Tripulantes', 'Lesões Leves Tripulantes')

In [0]:
classificacoes_excluir = ["Indeterminado", "Sem Registro", "Exterior"]

df_gold = df_gold.filter(~df_gold['Estado'].isin(classificacoes_excluir))

In [0]:
from pyspark.sql.functions import current_timestamp, date_format, from_utc_timestamp, to_timestamp, col, when, length, concat, lit

df_gold = df_gold \
    .withColumn("Atualização", date_format(from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"), date_hour_format_br)) \
    .withColumn("Data", date_format("Data", date_format_br)) \
    .withColumn("Hora", when(length(col("Hora")) == 5, concat(col("Hora"), lit(":00"))).otherwise(col("Hora"))
)


In [0]:
df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .option("delta.columnMapping.mode", "name") \
    .save(f'{gold_path}' + "ocorrencia_ampla")

In [0]:
%sql

CREATE TABLE IF NOT EXISTS anac.analytics.gold_ocorrencia_ampla

LOCATION 'abfss://anac@lfbazure.dfs.core.windows.net/gold/ocorrencia_ampla';